# Part 2 — SoRA gate: proximal soft-thresholding vs. L1-subgradient SGD

SoRA (Ding et al., 2023) trains the gate **g** with a **proximal gradient** step
$$g_{t+1}=\mathcal{T}_{\eta_t\lambda}\big(g_t-\eta_t\nabla_g\mathcal{L}_0(g_t)\big),\qquad \mathcal{T}_\xi(x)=\mathrm{sign}(x)\max(|x|-\xi,0).$$

This notebook replaces that with **plain SGD on the L1 subgradient**
$$g_{t+1}=g_t-\eta_t\big(\nabla_g\mathcal{L}_0(g_t)+\lambda\,\partial\|g_t\|_1\big),\qquad \partial\|g\|_1=\mathrm{sign}(g),$$
implements **both** in numpy and PyTorch (forward + backward), and compares them
to each other and to the proximal update from Part 1.

**Isolating the gate.** A SoRA layer's update is `h = B diag(g) A x`. The rank-`j`
contribution is `g_j * f_j(x)` with `f_j(x)=(A[j]·x)·B[:,j]`, so the prediction is
**linear in g**. Fitting `g` is therefore exactly a LASSO problem
`L0(g)=½‖Φg−y‖² , +λ‖g‖₁`, which isolates the gate dynamics from the rest of the
network. We plant a sparse ground-truth gate so we can check support recovery.

In [ ]:
import numpy as np, torch
rng = np.random.default_rng(0)
r, d, k, N = 8, 16, 4, 400          # rank, in-dim, out-dim, samples
A = rng.standard_normal((r, d)); B = rng.standard_normal((k, r))
X = rng.standard_normal((N, d))

# True sparse gate: only 3 of 8 ranks active -> ideal effective rank = 3
g_true = np.array([1.5, 0.0, -2.0, 0.0, 0.0, 0.8, 0.0, 0.0])

# Build Phi (N*k, r): column j is the vectorised rank-j feature f_j(x)
AX = X @ A.T                                   # (N, r)
Phi = np.zeros((N * k, r))
for j in range(r):
    Phi[:, j] = np.outer(AX[:, j], B[:, j]).reshape(-1)
y = Phi @ g_true + 0.01 * rng.standard_normal(N * k)

lam, eta, steps = 0.05, 0.02, 4000
print("data ready: Phi", Phi.shape, "| true effective rank =", int((g_true!=0).sum()))

data ready: Phi (1600, 8) | true effective rank = 3


## 1. numpy implementation (manual forward **and** backward)

`L0(g) = ½·mean((Φg − y)²)`  → analytic gradient `∇L0(g) = (1/M)·Φᵀ(Φg − y)`.
We implement the smooth gradient by hand (the "backward pass") and both gate
update rules on top of it.

In [ ]:
def grad_L0_np(g):                       # manual backward pass for the smooth part
    return Phi.T @ (Phi @ g - y) / len(y)

def soft_threshold_np(x, t):             # proximal operator of t·|·|
    return np.sign(x) * np.maximum(np.abs(x) - t, 0.0)

def proximal_np(steps=steps):            # Part 1 update
    g = np.zeros(r)
    for _ in range(steps):
        g = soft_threshold_np(g - eta * grad_L0_np(g), eta * lam)
    return g

def subgrad_sgd_np(steps=steps, sign0=0.0):   # Part 2 update
    g = np.zeros(r)
    for _ in range(steps):
        s = np.sign(g); s[g == 0] = sign0     # subgradient choice at exactly 0
        g = g - eta * (grad_L0_np(g) + lam * s)
    return g

g_prox_np = proximal_np()
g_sgd_np  = subgrad_sgd_np()
np.set_printoptions(precision=4, suppress=True)
print("proximal (np):", g_prox_np)
print("sgd      (np):", g_sgd_np)

proximal (np): [ 1.4952  0.     -1.9995 -0.      0.      0.7958  0.      0.    ]
sgd      (np): [ 1.495   0.0009 -1.9995  0.0003  0.0008  0.7958  0.     -0.0001]


## 2. PyTorch implementation (autograd backward **and** manual gate step)

Same two rules, but the smooth gradient `∇L0` now comes from autograd, to show the
gate-update logic is independent of how the gradient is obtained.

In [ ]:
Phi_t = torch.tensor(Phi); y_t = torch.tensor(y)

def grad_L0_torch(g):                    # autograd backward pass
    g = g.clone().detach().requires_grad_(True)
    (0.5 * ((Phi_t @ g - y_t) ** 2).mean()).backward()
    return g.grad.detach()

def proximal_torch(steps=steps):
    g = torch.zeros(r, dtype=torch.float64)
    for _ in range(steps):
        z = g - eta * grad_L0_torch(g)
        g = torch.sign(z) * torch.clamp(z.abs() - eta * lam, min=0.0)
    return g.numpy()

def subgrad_sgd_torch(steps=steps, sign0=0.0):
    g = torch.zeros(r, dtype=torch.float64)
    for _ in range(steps):
        s = torch.sign(g); s[g == 0] = sign0
        g = g - eta * (grad_L0_torch(g) + lam * s)
    return g.numpy()

g_prox_torch = proximal_torch()
g_sgd_torch  = subgrad_sgd_torch()
print("proximal (torch):", g_prox_torch)
print("sgd      (torch):", g_sgd_torch)

proximal (torch): [ 1.4952  0.     -1.9995 -0.      0.      0.7958  0.      0.    ]
sgd      (torch): [ 1.495   0.0009 -1.9995  0.0003  0.0008  0.7958  0.     -0.0001]


## 3. Tests — (i) numpy vs PyTorch agree, (ii) backward pass is correct

In [ ]:
# (i) the two implementations must match
print("PROX  numpy-vs-torch max|diff| =", np.abs(g_prox_np - g_prox_torch).max())
print("SGD   numpy-vs-torch max|diff| =", np.abs(g_sgd_np  - g_sgd_torch ).max())

# (ii) backward-pass check: manual analytic gradient == autograd gradient
g0 = rng.standard_normal(r)
err = np.abs(grad_L0_np(g0) - grad_L0_torch(torch.tensor(g0)).numpy()).max()
print("backward check  max|grad_np - grad_torch| =", err)

assert np.abs(g_prox_np - g_prox_torch).max() < 1e-10
assert np.abs(g_sgd_np  - g_sgd_torch ).max() < 1e-10
assert err < 1e-8
print("\nAll tests passed ✓")

PROX  numpy-vs-torch max|diff| = 0.0
SGD   numpy-vs-torch max|diff| = 1.1102230246251565e-16
backward check  max|grad_np - grad_torch| = 1.9539925233402755e-14

All tests passed ✓


## 4. Comparison with Part 1 (proximal): the sparsity difference

The decisive metric is **how many gates are *exactly* zero**, because only exact
zeros let SoRA prune ranks (reduce effective rank).

In [ ]:
eps = 1e-8
def report(name, g):
    nz = int((np.abs(g) >= eps).sum())
    print(f"{name:28s} exact_zeros={int((np.abs(g)<eps).sum())}/{r}  "
          f"effective_rank={nz}  gate@true-zeros={g[[1,3,4,6,7]]}")

print("true effective rank = 3\n")
report("Proximal (soft-threshold)", g_prox_np)
report("SGD (L1 subgradient)",      g_sgd_np)

true effective rank = 3

Proximal (soft-threshold)    exact_zeros=5/8  effective_rank=3  gate@true-zeros=[ 0. -0.  0.  0.  0.]
SGD (L1 subgradient)         exact_zeros=0/8  effective_rank=8  gate@true-zeros=[ 0.0009  0.0003  0.0008  0.     -0.0001]


**Result.** Proximal drives 5 gates to *exactly* 0 → recovers the true rank-3
support. Subgradient SGD shrinks the same gates to tiny values (≈1e-3) but **never
exactly zero**, so its effective rank stays at the full 8. See the writeup for why.

## 5. Part (c) — does the subgradient choice at 0 matter?

In [ ]:
for s0, label in [(0.0,"sign(0)=0"), (1.0,"sign(0)=+1")]:
    g = subgrad_sgd_np(sign0=s0)
    print(f"{label:12s} exact_zeros={int((np.abs(g)<eps).sum())}/{r}  gate@true-zeros={g[[1,3,4,6,7]]}")
print("\nObservation: the at-0 convention barely matters — SGD almost never lands")
print("exactly on 0, so the subgradient-at-0 value is essentially never used.")

sign(0)=0    exact_zeros=0/8  gate@true-zeros=[ 0.0009  0.0003  0.0008  0.     -0.0001]
sign(0)=+1   exact_zeros=0/8  gate@true-zeros=[ 0.0009  0.0003 -0.0003  0.     -0.0001]

Observation: the at-0 convention barely matters — SGD almost never lands
exactly on 0, so the subgradient-at-0 value is essentially never used.
